In [4]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd


# Load data
result_path = "data/"

train_traded_volume_df = pd.read_csv(
    os.path.join(result_path, "train_traded_volume_201901_20.csv"),
    index_col=[0, 1]
)

test_traded_volume_df = pd.read_csv(
    os.path.join(result_path, "test_traded_volume_201902_20.csv"),
    index_col=[0, 1]
)

train_px_df = pd.read_csv(
    os.path.join(result_path, "train_px_201901_20.csv"),
    index_col=[0, 1]
)

test_px_df = pd.read_csv(
    os.path.join(result_path, "test_px_201902_20.csv"),
    index_col=[0, 1]
)


In [3]:
def detect_possible_splits(px_df, traded_volume_df, price_tol=0.08, volume_tol=0.35):
    """
    Detect possible stock splits using overnight price jumps and daily volume jumps.

    px_df:
        index = (stock, date)
        columns = intraday time bins
        values = prices

    traded_volume_df:
        same index/columns
        values = traded volume
    """

    rows = []

    first_col = px_df.columns[0]
    last_col = px_df.columns[-1]

    daily_open = px_df[first_col].astype(float)
    daily_close = px_df[last_col].astype(float)
    daily_volume = traded_volume_df.sum(axis=1).astype(float)

    stocks = px_df.index.get_level_values("stock").unique()

    split_ratios = [2, 3, 4, 5, 10]

    for stock in stocks:
        stock_open = daily_open.loc[stock].sort_index()
        stock_close = daily_close.loc[stock].sort_index()
        stock_volume = daily_volume.loc[stock].sort_index()

        prev_close = stock_close.shift(1)
        prev_volume = stock_volume.shift(1)

        price_ratio = stock_open / prev_close
        volume_ratio = stock_volume / prev_volume
        notional_ratio = (stock_open * stock_volume) / (prev_close * prev_volume)

        for date in stock_open.index:
            if pd.isna(price_ratio.loc[date]) or pd.isna(volume_ratio.loc[date]):
                continue

            for ratio in split_ratios:
                forward_split = (
                    abs(price_ratio.loc[date] - 1 / ratio) < price_tol
                    and abs(volume_ratio.loc[date] - ratio) < volume_tol * ratio
                )

                reverse_split = (
                    abs(price_ratio.loc[date] - ratio) < price_tol * ratio
                    and abs(volume_ratio.loc[date] - 1 / ratio) < volume_tol
                )

                if forward_split or reverse_split:
                    rows.append({
                        "stock": stock,
                        "date": date,
                        "possible_split_ratio": ratio,
                        "split_type": "forward" if forward_split else "reverse",
                        "price_ratio_open_to_prev_close": price_ratio.loc[date],
                        "volume_ratio_to_prev_day": volume_ratio.loc[date],
                        "notional_ratio": notional_ratio.loc[date],
                        "prev_close": prev_close.loc[date],
                        "open": stock_open.loc[date],
                        "prev_volume": prev_volume.loc[date],
                        "volume": stock_volume.loc[date],
                    })

    return pd.DataFrame(rows)

In [5]:
train_possible_splits_df = detect_possible_splits(
    px_df=train_px_df,
    traded_volume_df=train_traded_volume_df
)

test_possible_splits_df = detect_possible_splits(
    px_df=test_px_df,
    traded_volume_df=test_traded_volume_df
)

display(train_possible_splits_df)
display(test_possible_splits_df)

""


""
